## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
from src.algorithms import AvgConsensus, RatioConsensus
from src.problem import create_consensus_problem

import decent_bench.utils.interoperability as iop
from decent_bench import benchmark
from decent_bench.agents import Agent
from decent_bench.algorithms.p2p import ADMM
from decent_bench.algorithms.utils import normal_initialization
from decent_bench.metrics import metric_library as ml
from decent_bench.networks import P2PNetwork
from decent_bench.schemes import GaussianNoise, Quantization, UniformActivationRate, UniformDropRate
from decent_bench.utils.checkpoint_manager import CheckpointManager
from decent_bench.utils.network_utils import plot_network


## Experiment parameters

In [ ]:
checkpoint_path = Path("results")

iterations = 2_000
n_trials = 3
n_agents = 5
n_neighbors = 4
x_size = 10

# Define network impairments, set to None to disable
activation = UniformActivationRate(0.8)
noise = GaussianNoise(0.0, 0.01)
compression = Quantization(0.1)
drops = UniformDropRate(0.2)

# Set seed
iop.set_seed(47)

## Define which metrics to calculate

In [ ]:
table_metrics = [
    ml.ConsensusError(),
    ml.SentMessages(),
    ml.ReceivedMessages(),
    ml.SentMessagesDropped(),
]

plot_metrics = [
    ml.ConsensusError(),
]

## Create checkpoint manager

In [ ]:
cm = CheckpointManager(
    checkpoint_dir=checkpoint_path,
    checkpoint_step=None,
    keep_n_checkpoints=1,
    benchmark_metadata={
        "dataset": "consensus",
        "n_agents": n_agents,
        "n_neighbors": n_neighbors,
        "drops": drops.__class__.__name__ if drops else None,
        "activity": activation.__class__.__name__ if activation else None,
        "compression": compression.__class__.__name__ if compression else None,
        "noise": noise.__class__.__name__ if noise else None,
    },
)

## Create costs, agents, network, and benchmark problem

In [ ]:
costs, x_optimal, u = create_consensus_problem(x_size, n_agents)

agents = [
    Agent(
        costs[i],
        activation=activation,
        data={"u": u[i]}  # store local observations, needed by AvgConsensus, RatioConsensus
    )
    for i in range(n_agents)
]
graph = nx.random_regular_graph(d=n_neighbors, n=n_agents, seed=iop.get_seed())
network = P2PNetwork(
    graph=graph,
    agents=agents,
    message_noise=noise,
    message_compression=compression,
    message_drop=drops,
)
problem = benchmark.BenchmarkProblem(
    network=network,
    x_optimal=x_optimal,
)

### Plot network

In [ ]:
fig = plt.figure()
plot_network(network, ax=fig.gca())
plt.show()

## Create algorithms

In [ ]:
x0 = normal_initialization(network)
algorithms = [
    ADMM(
        iterations=iterations,
        penalty=10,
        relaxation=0.5,
        x0=x0,
    ),
    AvgConsensus(
        iterations=iterations,
    ),
    RatioConsensus(
        iterations=iterations,
    ),
]

## Run benchmark

In [ ]:
result = benchmark.benchmark(
    algorithms=algorithms,
    benchmark_problem=problem,
    n_trials=n_trials,
    show_speed=True,
    show_trial=True,
    checkpoint_manager=cm,
)

metric_result = benchmark.compute_metrics(
    benchmark_result=result,
    checkpoint_manager=cm,
    table_metrics=table_metrics,
    plot_metrics=plot_metrics,
)

benchmark.display_metrics(
    metrics_result=metric_result,
    checkpoint_manager=cm,
)